# Lekshan 10 - AI Agents for Production

For dis lekshan, you go learn **production patterns** for AI agents wen dem dey use Microsoft Agent Framework (Python). We go cover:

- **Observability** — to add timing and logging to how agent dem dey interact
- **Evaluation** — to use evaluator agent to score quality of response dem
- **Cost management** — strategy them for token optimization and how to choose model

Di tin we dey talk na **travel agent** wey dey help people plan their trips, plus monitoring and evaluation wey dey on top.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Consideration for Production

To move AI agents from prototypes go production, you for pay beta attention to three main tins:

1. **Observability** — You need make you fit see wetin the agent dey do, how long e dey take, and which tools e dey call. If you no get tracing and logging, e go hard to find and solve production wahala.

2. **Evaluation** — Automated quality checks dey make sure say the agent response still dey correct, complete, and useful over time. One evaluator agent fit score the response based on set criteria dem.

3. **Cost Management** — How you take use token go affect cost. Strategies like prompt optimization, how you select model, and caching dey help to keep expenses under control without losing quality.


## Di way to build an Observable Agent

We dey define travel tools and we wrap di agent call with timing so dat we fit monitor latency. For production, you go join am with OpenTelemetry or sometin wey similar for tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

One common way wey dem dey do am for production na to use second person wey be **evaluator**. The evaluator dey score the main agent answer based on criteria wey dem don set like if e complete, if e correct, and if e dey helpful.

This one dey enable:
- Automated quality gates before response dem reach users
- Regression detection anytime prompts or models change
- Continuous monitoring of how agent dey perform over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Cost Management Strategies

Managing cost na very important tin for production AI agents. Below na di main strategies:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Make system instructions short and clear. Cut out any repeat context to reduce input tokens. |
    "| **Model selection** | Use smaller, cheaper models (like GPT-5-mini) for simple work like classification or extraction, and keep big models for heavy reasoning work. |\n",
| **Caching** | Save tool results and common queries to avoid doing same API calls again. |
| **Token budgets** | Set `max_tokens` limits to make sure responses no too long unexpectedly. |
| **Batching** | Join many user queries inside one API call when e fit work. |

For real life, tiered way dey successful: send easy requests go fast, cheap model and only send complex questions to better model. 


## Summary

For dis lesson, you learn how to:

1. **Add observability** to agent interactions wit timing and logging, wey go lay di groundwork for tracing and monitoring.
2. **Evaluate agent responses** automatically using one evaluator agent wey dey score completeness, accuracy, and helpfulness.
3. **Manage costs** through prompt optimization, model selection, caching, and token budgets.

Dem production patterns nawetin dey help make sure sey your AI agents dey reliable, measurable, and cost-effective as e dey grow.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
